# QLoRA fine-tune Qwen2.5-3B trên Kaggle (quota riêng Colab, chạy nền 12h)
**Settings** (panel phải): Accelerator = **GPU T4 x2** (hoặc P100) · Internet = **On**.

**Chạy NỀN (khuyến nghị):** bấm **Save Version → Save & Run All (Commit)** → notebook chạy headless tới ~12h, không cần canh, không lo ngắt. Adapter lưu ở `/kaggle/working/qwen-lora` (tải về sau ở tab Output).

Self-host ≤9B, KHÔNG API ngoài. Cấu hình T4-lite ~2-3h.

In [ ]:
# 1) Lấy code
%cd /kaggle/working
![ -d viettel-medreason ] || git clone https://github.com/tienndat1904/viettel-medreason.git
%cd /kaggle/working/viettel-medreason
!git pull -q

In [ ]:
# 2) Cài env — KHÔNG động vào numpy/transformers/torch (dùng bản Kaggle ship sẵn).
#    Chỉ update bitsandbytes/peft/accelerate.
!pip install -q -r requirements.txt
!pip install -q -U bitsandbytes peft accelerate
import numpy, torch, transformers, bitsandbytes as bnb, peft
print('numpy', numpy.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__, '| bnb', bnb.__version__, '| peft', peft.__version__, '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if not numpy.__version__.startswith('2'):
    raise SystemExit('numpy != 2.x -> Restart & Run All')

In [ ]:
# 3) SMOKE-TRAIN vài bước (kiểm không OOM). CUDA_VISIBLE_DEVICES=0 -> dùng 1 GPU cho ổn định.
import os; os.environ['CUDA_VISIBLE_DEVICES'] = '0'
MODEL = 'Qwen/Qwen2.5-3B-Instruct'   # OOM/quá chậm -> 'Qwen/Qwen2.5-1.5B-Instruct'
!python scripts/train_qlora.py --data data/synthetic/train_sft.jsonl --model {MODEL} --out /kaggle/working/smoke-lora --epochs 0.05 --max-len 2048 --seed 42

In [ ]:
# 4) TRAIN — T4-lite (~2-3h): 500 mẫu, 1 epoch, max_len 1536. Adapter -> /kaggle/working/qwen-lora.
#    Muốn full (chất lượng cao hơn): bỏ --max-samples, --epochs 2, --max-len 2048.
ADAPTER = '/kaggle/working/qwen-lora'
MODEL = 'Qwen/Qwen2.5-3B-Instruct'
!python scripts/train_qlora.py --data data/synthetic/train_sft.jsonl --model {MODEL} --out {ADAPTER} --max-samples 500 --epochs 1 --batch 1 --grad-accum 8 --lr 2e-4 --max-len 2048 --seed 42
!ls -la {ADAPTER}

In [ ]:
# 5) Trỏ config sang adapter + backend llm (inference tự bỏ few-shot khi có adapter)
import yaml
ADAPTER = '/kaggle/working/qwen-lora'
cfg = yaml.safe_load(open('configs/config.yaml', encoding='utf-8'))
cfg['extract']['backend'] = 'llm'
cfg['extract']['lora_adapter'] = ADAPTER
cfg['extract']['llm_model'] = 'Qwen/Qwen2.5-3B-Instruct'
yaml.safe_dump(cfg, open('configs/config.yaml', 'w', encoding='utf-8'), allow_unicode=True, sort_keys=False)
print('lora_adapter =', cfg['extract']['lora_adapter'])

In [ ]:
# 6) Đo QLoRA trên 30 file dev (METRIC CHÍNH THỨC) — so với few-shot / rule
import os, shutil, glob
os.makedirs('devset/input', exist_ok=True)
for g in glob.glob('data/dev/gold/*.json'):
    n = os.path.basename(g)[:-5]
    shutil.copy(f'data/test/input/{n}.txt', f'devset/input/{n}.txt')
!python src/pipeline.py --input devset/input --output dev_pred --backend llm
!python src/eval/official_scorer.py --pred dev_pred --gold data/dev/gold_who   # BTC-realistic (ICD WHO)
!python src/eval/official_scorer.py --pred dev_pred --gold data/dev/gold

In [ ]:
# 7) (Khi hài lòng) chạy full 100 file + đóng gói. output.zip + qwen-lora ở /kaggle/working (tab Output).
!python src/pipeline.py --input data/test/input --output /kaggle/working/vmr_output --backend llm
!python scripts/package_submission.py --output /kaggle/working/vmr_output --input data/test/input --n 100
!cp output.zip /kaggle/working/output.zip 2>/dev/null; ls -la /kaggle/working/*.zip